# MalWeave VERA DiE

Run a fresh packing audit on all 38,282 original PEs in the pinned VERA
inventory. Each worker stages one verified PE, runs `file` and then
`recursive`, `deep`, and `heuristic` in order using portable DiE 3.21 and
a 10-second timeout per mode.

Use a private CPU notebook with Internet enabled. Attach scoped AWS Secrets
`AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY`, and `AWS_SESSION_TOKEN` for
temporary credentials. The credentials need `s3:ListBucket` for the release
check and `s3:GetObject` for the staged bundle and original PE objects.

The default `LIMIT = None` scans the entire inventory. Set a positive limit for
a bounded run. Use **Save Version → Save & Run All** to save the SQLite
checkpoint, console log, summary, tool identity and release report in
`vera-die/`. A complete attempt over the release also exports `packing.csv`.
Attach the saved output of an earlier packing run to resume.


In [ ]:
DATASET = "vera"
LIMIT = None
import os
from pathlib import Path

WORKERS = 16
SESSION_HOURS = 10
RESUME_DB = ""  # Optional path to a state.sqlite3 mounted as a Kaggle input.
INPUT_BUNDLE = None  # Imported copies can set MALWEAVE_VERA_DIE_INPUTS as a JSON secret.

STAGING = Path("/kaggle/temp/malweave-vera-die")
OUTPUT = Path("/kaggle/working/vera-die")
REGION = "ap-southeast-1"
PYTHON = "3.12.12"
BOTO3 = "1.43.123"


In [ ]:
import hashlib
import json
import shutil
import subprocess
import tarfile
from urllib.request import urlopen

from kaggle_secrets import UserSecretsClient

os.umask(0o077)
STAGING.mkdir(parents=True, exist_ok=True)
secrets = UserSecretsClient()
bundle = INPUT_BUNDLE or json.loads(secrets.get_secret("MALWEAVE_VERA_DIE_INPUTS"))
env = dict(
    os.environ,
    AWS_DEFAULT_REGION=REGION,
    MALWEAVE_S3_ACCOUNT_ID=secrets.get_secret("MALWEAVE_S3_ACCOUNT_ID"),
    AWS_EC2_METADATA_DISABLED="true",
    UV_CACHE_DIR=str(STAGING / "uv-cache"),
    UV_PYTHON_INSTALL_DIR=str(STAGING / "python"),
    UV_PROJECT_ENVIRONMENT=str(STAGING / "env"),
    PYTHONUNBUFFERED="1",
)
env.pop("VIRTUAL_ENV", None)
for name in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"):
    env[name] = secrets.get_secret(name)
try:
    env["AWS_SESSION_TOKEN"] = secrets.get_secret("AWS_SESSION_TOKEN")
except Exception:
    env.pop("AWS_SESSION_TOKEN", None)

def sha256_file(path):
    with path.open("rb") as source:
        return hashlib.file_digest(source, "sha256").hexdigest()

UV_SHA256 = "5c43f82077ff0cd5aec588286cbabd89913e4d045bd4e8aa60b20b3ecffc36e3"
UV = STAGING / "uv"
if not UV.is_file():
    archive = STAGING / "uv.tar.gz"
    with (
        urlopen("https://github.com/astral-sh/uv/releases/download/0.11.9/"
                "uv-x86_64-unknown-linux-gnu.tar.gz", timeout=120) as source,
        archive.open("wb") as target,
    ):
        shutil.copyfileobj(source, target)
    if sha256_file(archive) != UV_SHA256:
        raise RuntimeError("Downloaded uv archive failed SHA-256 verification.")
    with tarfile.open(archive, "r:gz") as source:
        member = source.getmember("uv-x86_64-unknown-linux-gnu/uv")
        with source.extractfile(member) as binary, UV.open("wb") as target:
            shutil.copyfileobj(binary, target)
    UV.chmod(0o700)
    archive.unlink()

subprocess.run([str(UV), "python", "install", PYTHON], env=env, cwd=STAGING, check=True)
input_archive = STAGING / "audit-inputs.tar.gz"
if not input_archive.is_file() or sha256_file(input_archive) != bundle["sha256"]:
    temporary = input_archive.with_suffix(".download")
    temporary.unlink(missing_ok=True)
    download = """
import sys
import boto3
boto3.client("s3").download_file(*sys.argv[1:4])
"""
    subprocess.run(
        [str(UV), "run", "--no-project", "--python", PYTHON, "--with", f"boto3=={BOTO3}",
         "python", "-c", download, bundle["bucket"], bundle["key"], str(temporary)],
        env=env, cwd=STAGING, check=True,
    )
    if temporary.stat().st_size != bundle["bytes"] or sha256_file(temporary) != bundle["sha256"]:
        raise RuntimeError("VERA input bundle failed size or SHA-256 verification.")
    temporary.replace(input_archive)

with tarfile.open(input_archive, "r:gz") as source:
    members = source.getmembers()
    if any(not (member.isfile() or member.isdir()) for member in members):
        raise RuntimeError("Expected only regular files and directories in the VERA source bundle.")
    source.extractall(STAGING, members=members, filter="data")

REPO = STAGING / "repo"
print("Verified VERA source bundle. The audit runner reads the published S3 inventory.", flush=True)

import re

setup_command = [
    str(UV), "run", "--locked", "--python", PYTHON, "--no-default-groups",
    "malweave", "data", "die-setup",
]
initial = subprocess.run(setup_command, env=env, cwd=REPO)
if initial.returncode:
    portable = REPO / "data/external/tools/die-portable-3.21/die_linux_portable/base"
    die = portable / "diec"
    loader_env = dict(env, LD_LIBRARY_PATH=os.pathsep.join(
        filter(None, (str(portable), env.get("LD_LIBRARY_PATH")))
    ))
    diagnostics = subprocess.run(["ldd", str(die)], env=loader_env, capture_output=True, text=True)
    missing = sorted(set(re.findall(r"^\s*(\S+)\s*=>\s*not found", diagnostics.stdout, re.M)))
    if not missing:
        probe = subprocess.run([str(die), "--version"], env=loader_env, capture_output=True, text=True)
        raise RuntimeError(f"DiE setup failed: {probe.stderr.strip() or diagnostics.stderr.strip()}")
    print("Portable DiE missing system libraries:", ", ".join(missing), flush=True)

    packages = {
        "libdouble-conversion.so.3": ("libdouble-conversion3",),
        "libatomic.so.1": ("libatomic1",),
        "libpcre2-16.so.0": ("libpcre2-16-0",),
        "libpcre2-8.so.0": ("libpcre2-8-0",),
        "libglib-2.0.so.0": ("libglib2.0-0t64", "libglib2.0-0"),
        "libz.so.1": ("zlib1g",),
    }
    if unknown := set(missing) - packages.keys():
        raise RuntimeError("Unrecognized DiE system library: " + ", ".join(sorted(unknown)))

    apt = STAGING / "die-system-libraries"
    apt.mkdir(exist_ok=True)
    indexes = apt / "lists"
    indexes.mkdir(exist_ok=True)
    (indexes / "partial").mkdir(exist_ok=True)
    apt_flags = ["-o", f"Dir::State::lists={indexes}", "-o", "Acquire::Retries=1"]
    subprocess.run(["apt-get", *apt_flags, "update", "-qq"], cwd=apt, check=True, timeout=180)
    for library in missing:
        package = next((candidate for candidate in packages[library] if subprocess.run(
            ["apt-cache", *apt_flags, "show", candidate],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        ).returncode == 0), None)
        if package is None:
            raise RuntimeError(f"Kaggle package indexes do not contain a provider for {library}.")
        subprocess.run(["apt-get", *apt_flags, "download", package], cwd=apt, check=True, timeout=120)
        deb, = apt.glob(package + "_*.deb")
        unpacked = apt / package
        subprocess.run(["dpkg-deb", "-x", str(deb), str(unpacked)], check=True)
        found = next((path for path in unpacked.rglob(library) if path.is_file()), None)
        if found is None:
            raise RuntimeError(f"{package} did not contain {library}.")
        shutil.copyfile(found, portable / library)
    subprocess.run(setup_command, env=env, cwd=REPO, check=True)



In [ ]:
command = [
    str(UV), "run", "--locked", "--python", PYTHON, "--no-default-groups",
    "--with", f"boto3=={BOTO3}", "python", "scripts/run-dataset-audit-kaggle.py", "full",
    "--workflow", "dataset-die", "--dataset", DATASET, "--job", str(STAGING / "job"), "--output", str(OUTPUT),
    "--workers", str(WORKERS), "--session-seconds", str(SESSION_HOURS * 3600),
]
if LIMIT is not None:
    command.extend(["--limit", str(LIMIT)])

if RESUME_DB:
    checkpoint = Path(RESUME_DB)
    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)
else:
    attached = sorted(
        path for path in Path("/kaggle/input").rglob(f"{DATASET}-die/state.sqlite3")
        if path.is_file()
    )
    if len(attached) > 1:
        raise RuntimeError("Set RESUME_DB to the checkpoint you want to resume.")
    checkpoint = attached[0] if attached else STAGING / "resume/state.sqlite3"

if checkpoint.is_file():
    command.extend(["--restore", str(checkpoint)])

subprocess.run(command, env=env, cwd=REPO, check=True)
